In [1]:
import os
import shutil
import pandas as pd
from pathlib import Path

# ============================================================
# 1. CONFIG
# ============================================================

ROOT = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

OUTPUT = Path("/kaggle/working/RSNA_58_studies")

TRAIN_CSV = ROOT / "train.csv"
TRAIN_SERIES_CSV = ROOT / "train_series.csv"
TRAIN_SERIES_DIR = ROOT / "train_series"

OUTPUT_SERIES_DIR = OUTPUT / "train_series"
OUTPUT_METADATA_DIR = OUTPUT / "metadata"

OUTPUT_SERIES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_METADATA_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. ĐỌC TRAIN.CSV
# ============================================================

train = pd.read_csv(TRAIN_CSV)

label_cols = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

# Study có đầy đủ 12 nhãn
labeled = train[
    train[label_cols].notna().all(axis=1)
].copy()

study_ids = labeled["StudyInstanceUID"].tolist()

print("=" * 60)
print(f"Số study có đầy đủ nhãn: {len(study_ids)}")
print("=" * 60)


# ============================================================
# 3. LƯU DANH SÁCH 58 STUDY
# ============================================================

labeled.to_csv(
    OUTPUT_METADATA_DIR / "labeled_58.csv",
    index=False
)

print("Đã lưu:")
print(OUTPUT_METADATA_DIR / "labeled_58.csv")


# ============================================================
# 4. ĐỌC TRAIN_SERIES.CSV
# ============================================================

train_series = pd.read_csv(TRAIN_SERIES_CSV)

labeled_series = train_series[
    train_series["StudyInstanceUID"].isin(study_ids)
].copy()

labeled_series.to_csv(
    OUTPUT_METADATA_DIR / "labeled_58_series.csv",
    index=False
)

print(
    f"Số series thuộc 58 study: "
    f"{len(labeled_series):,}"
)


# ============================================================
# 5. COPY DICOM CỦA 58 STUDY
# ============================================================

print("\nBắt đầu copy DICOM...\n")

for i, study_id in enumerate(study_ids, start=1):

    src = TRAIN_SERIES_DIR / study_id
    dst = OUTPUT_SERIES_DIR / study_id

    if not src.exists():
        print(f"[WARNING] Không tìm thấy: {study_id}")
        continue

    shutil.copytree(
        src,
        dst,
        dirs_exist_ok=True
    )

    print(
        f"[{i:02d}/58] "
        f"{study_id}"
    )


# ============================================================
# 6. THỐNG KÊ SỐ SERIES / SỐ IMAGE
# ============================================================

stats = []

for study_id in study_ids:

    study_dir = OUTPUT_SERIES_DIR / study_id

    if not study_dir.exists():
        continue

    series_dirs = [
        p for p in study_dir.iterdir()
        if p.is_dir()
    ]

    dcm_files = list(study_dir.rglob("*.dcm"))

    stats.append({
        "StudyInstanceUID": study_id,
        "Num_Series": len(series_dirs),
        "Num_DICOM": len(dcm_files)
    })


stats_df = pd.DataFrame(stats)

stats_df.to_csv(
    OUTPUT_METADATA_DIR / "study_statistics.csv",
    index=False
)

print("\n" + "=" * 60)
print("THỐNG KÊ")
print("=" * 60)

print(stats_df)

print("\nTổng series:", stats_df["Num_Series"].sum())
print("Tổng DICOM:", stats_df["Num_DICOM"].sum())


# ============================================================
# 7. COPY THÊM TRAIN.CSV / TRAIN_SERIES.CSV
# ============================================================

shutil.copy(
    TRAIN_CSV,
    OUTPUT_METADATA_DIR / "train.csv"
)

shutil.copy(
    TRAIN_SERIES_CSV,
    OUTPUT_METADATA_DIR / "train_series.csv"
)


# ============================================================
# 8. NÉN THÀNH ZIP
# ============================================================

print("\nĐang tạo ZIP...")

zip_path = shutil.make_archive(
    "/kaggle/working/RSNA_58_studies",
    "zip",
    OUTPUT
)

print("\n" + "=" * 60)
print("HOÀN TẤT")
print("=" * 60)

print(f"ZIP: {zip_path}")

size_gb = os.path.getsize(zip_path) / (1024 ** 3)

print(f"Kích thước ZIP: {size_gb:.2f} GB")

Số study có đầy đủ nhãn: 58
Đã lưu:
/kaggle/working/RSNA_58_studies/metadata/labeled_58.csv
Số series thuộc 58 study: 336

Bắt đầu copy DICOM...

[01/58] 1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149
[02/58] 1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269
[03/58] 1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052
[04/58] 1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580
[05/58] 1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238
[06/58] 1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935
[07/58] 1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840
[08/58] 1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756
[09/58] 1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604
[10/58] 1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581
[11/58] 1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945
[12/58] 1.2.826.0.1.3680043.8.498.12505035424093604